In [11]:
# PREDICTION WITH TRAINED ANN MODEL
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [12]:
# load the trained model, scaler, pickle, onehot
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    Dense(64, activation='relu', input_shape=(12,)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.load_weights("model.weights.h5")

# load the encoders and scaler
with open('onehot_encoder_geo.pkl', 'rb')as file:
    label_encoder_geo=pickle.load(file)
    
with open('label_encoder_gender.pkl', 'rb')as file:
    label_encoder_gender=pickle.load(file)
    
with open('scaler.pkl', 'rb')as file:
    scaler=pickle.load(file)

c:\Users\Hrushi\Desktop\GenAI\DL\DLENV\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
# Example input data
input_data={
    'CreditScore':750,
    'Geography':'France',
    'Gender':'Male',
    'Age':45,
    'Tenure':2,
    'Balance':6000,
    'NumOfProducts':3,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':5000
}

In [14]:
# One-hot encode 'Geography'
geo_encoded=label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded, columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

c:\Users\Hrushi\Desktop\GenAI\DL\DLENV\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [15]:
# Here out input data is in the key_value format, so we have to convert it into dataframe before use
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,750,France,Male,45,2,6000,3,1,1,5000


In [16]:
# #Encode categorical Variables
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,750,France,1,45,2,6000,3,1,1,5000


In [17]:
# Concatination of one hot encoded
input_df=pd.concat([input_df.drop("Geography", axis=1), geo_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,750,1,45,2,6000,3,1,1,5000,1.0,0.0,0.0


In [18]:
# Scaling the input data
input_scaled=scaler.transform(input_df)
input_scaled

array([[ 1.03021428,  0.91360328,  0.58261671, -1.03249572, -1.12585605,
         2.54529536,  0.64506294,  0.96934842, -1.66205347,  1.00262813,
        -0.57961128, -0.57710974]])

In [19]:
# Prediction or predict the churn
prediction=model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step


array([[0.44993916]], dtype=float32)

In [20]:
# probaility of the prediction
prediction_prob= prediction[0][0]
prediction_prob

np.float32(0.44993916)

In [21]:
if prediction_prob>0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is not likely to churn")
    

The customer is not likely to churn
